### **Orbital Debris Database**   
The Orbital Debris database combines the cleaned UCS and SATCAT datasets into a single analytical source. All de-orbited objects have been removed.   

This notebook builds kinetic_master.csv into a SQLite database using Python, Pandas, and sqlite3, which will be used to run project queries and generate visualizations.

For database design, every table should have a unique primary key. In this project, each satellite is guaranteed to have a unique `norad_id`—there are never duplicates, and any are dropped during cleaning. This allows `norad_id` to serve as the primary key for the four main satellite-related tables. The `launch_events` and `ownership_operators` tables use `launch_id` and `owner_code` as their unique primary keys, respectively. This approach ensures data integrity and efficient joins, without the need for surrogate keys unless the data model changes in the future.

In [1]:
import pandas as pd
import sqlite3
import utility as utils

df_master = pd.read_csv('../data/clean/kinetic_master.csv', low_memory=False)

df = df_master.copy()

conn = sqlite3.connect('../data/clean/orbital_debris.db')

In [2]:
# Now we have to create a new SQLite database and write the dataframe to it.
# We need to separate the dataframe into multiple tables to avoid redundancy and to make it easier to query later on.
# We also want to patch some of the data that is missing or inconsistent, especially in the ownership metadata, 
# to create a clean ownership_operators table that we can join on later.
# We could have patched this in to the kinetic_master.csv directly, but doing it here allows us to keep the raw 
# master data intact and keeps the data transformation logic in one place. I am still considering breaking each table into
# separate notebooks where we patch the data and export a csv.  Then a final notebook to load the cleaned csvs
# into SQLite. For now, for simplicity, well do it all in this notebook.

# strip white space and standardize case for owner_code.
df['owner_code'] = df['owner_code'].astype(str).str.strip().str.upper()

# strip white space for owner and then standardize common variations of SpaceX to a single canonical name.
df['owner'] = df['owner'].astype(str).str.strip()

owner_name_map = {
    'Spacex': 'SpaceX',
    'spacex': 'SpaceX',
    'spaceX': 'SpaceX',
    'Swarm Technologies': 'SpaceX',
    'Space Exploration Technologies Corp.': 'SpaceX'
}

df['owner'] = df['owner'].replace(owner_name_map)

# aggregate ownership metadata by owner_code
# This will help us create an ownership_operators table without duplicates, 
# and we can then join on the satellite table using owner_code as a foreign key.

# coerce flag columns to numeric, filling non-convertable values with 0, then convert them to int. 
# This ensures that we have consistent 0/1 values for the boolean flags,
flag_cols = ['is_commercial', 'is_government', 'is_military', 'is_civil']

for col in flag_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# group by owner_code puts all rows for the same owner together then we aggregate the ownership metadata 
# by taking the first non-null value for string fields, and max for boolean flags.  This helps preserve 
# the most complete metadata for each owner while eliminating duplicates. 
owner_profile = df.groupby('owner_code', as_index=False).agg({
    'owner': utils.first_non_null,
    'country_operator': utils.first_non_null,
    'users': utils.first_non_null,
    'is_commercial': 'max',
    'is_government': 'max',
    'is_military': 'max',
    'is_civil': 'max',
    'contractor': utils.first_non_null,
    'contractor_country': utils.first_non_null
})

# only mark non-payload rows as Not Applicable
# this mask could be done a bit more simplier, is_payload = df['object_type'] == 'PAYLOAD'
# but i prefer this way to protect from case drift or accidental whitespace in the object_type field
# which would cause the simpler mask to fail.
#.eq('PAYLOAD') is equivalent to == 'PAYLOAD'.

is_payload = df['object_type'].astype(str).str.strip().str.upper().eq('PAYLOAD')

df.loc[~is_payload, 'primary_purpose'] = df.loc[~is_payload, 'primary_purpose'].fillna('Not Applicable')
df.loc[~is_payload, 'un_registry'] = df.loc[~is_payload, 'un_registry'].fillna('Not Applicable')

# keep lifetime_years numeric + nullable (no forced imputation)
df['lifetime_years'] = pd.to_numeric(df['lifetime_years'], errors='coerce')

# keep orbit_type stable
df['orbit_type'] = df['orbit_type'].fillna('Other/Misc')

# launch_id is synthetic: derive from COSPAR prefix YYYY-NNN
df['launch_id'] = df['cospar_id'].astype(str).str.extract(r'^(\d{4}-\d{3})', expand=False)

df['launch_id'] = df['launch_id'].fillna('UNKNOWN')

In [3]:
# build the individual tables for SQLite export, selecting relevant columns and dropping duplicates where necessary.
df_ownership_operators = owner_profile[
    ['owner_code', 'owner', 'country_operator', 'users',
     'is_commercial', 'is_government', 'is_military', 'is_civil',
     'contractor', 'contractor_country']
 ]

df_launch_events = df[
    ['launch_id', 'launch_date', 'launch_year', 'launch_site']
].drop_duplicates(subset=['launch_id'])


df_satellites = df[
    ['norad_id', 'cospar_id', 'object_name', 'satellite_name', 'official_name',
     'object_type', 'category', 'ops_status', 'data_status', 'in_orbit',
     'owner_code', 'launch_id']
].drop_duplicates(subset=['norad_id'])

df_orbital_data = df[
    ['norad_id', 'orbit_class', 'orbit_type', 'period_minutes', 'perigee_km',
     'apogee_km', 'inclination_degrees', 'eccentricity', 'semi_major_axis_km',
     'launch_mass_kg', 'proxy_mass_kg', 'dry_mass_kg', 'power_watts',
     'proxy_power_watts', 'rcs', 'rcs_class']
].drop_duplicates(subset=['norad_id'])

df_ucs_details = df[
    ['norad_id', 'lifetime_years', 'sat_age_years',
     'primary_purpose', 'detailed_purpose', 'geo_longitude', 'un_registry']
].drop_duplicates(subset=['norad_id'])

df_risk_assessment = df[
    ['norad_id', 'velocity_kms', 'kinetic_joules', 'is_zombie']
].drop_duplicates(subset=['norad_id'])

# write each dataframe to SQLite using schema-aligned table names
df_satellites.to_sql('satellites', conn, if_exists='replace', index=False)
df_orbital_data.to_sql('orbital_data', conn, if_exists='replace', index=False)
df_ucs_details.to_sql('ucs_details', conn, if_exists='replace', index=False)
df_risk_assessment.to_sql('risk_assessment', conn, if_exists='replace', index=False)
df_ownership_operators.to_sql('ownership_operators', conn, if_exists='replace', index=False)
df_launch_events.to_sql('launch_events', conn, if_exists='replace', index=False)

# sanity check of completed database.

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)['name']

# primary key columns for each table
primary_keys = {
    'satellites': 'norad_id',
    'orbital_data': 'norad_id',
    'ucs_details': 'norad_id',
    'risk_assessment': 'norad_id',
    'ownership_operators': 'owner_code',
    'launch_events': 'launch_id'
}

for table in tables:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    key_col = primary_keys.get(table)
    utils.quick_report(df, title=f"Table: {table}", key_col=key_col)

conn.commit()
conn.close()

print('\nSQLite build complete: ../data/clean/orbital_debris.db')

# Table: satellites

**Dimensions:** 33,358 rows × 12 columns

**Memory Footprint:** 16.10 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **cospar_id** | `object` | 0 | 100.0% | ✅ |
| **object_name** | `object` | 0 | 100.0% | ✅ |
| **satellite_name** | `object` | 27,853 | 16.5% | ⚠️ |
| **official_name** | `object` | 27,853 | 16.5% | ⚠️ |
| **object_type** | `object` | 0 | 100.0% | ✅ |
| **category** | `object` | 0 | 100.0% | ✅ |
| **ops_status** | `object` | 0 | 100.0% | ✅ |
| **data_status** | `object` | 32,414 | 2.8% | ⚠️ |
| **in_orbit** | `int64` | 0 | 100.0% | ✅ |
| **owner_code** | `object` | 0 | 100.0% | ✅ |
| **launch_id** | `object` | 0 | 100.0% | ✅ |

### 📝 Object Overview
|                |   count |   unique | top              |   freq |
|:---------------|--------:|---------:|:-----------------|-------:|
| cospar_id      |   33358 |    33358 | 1958-002B        |      1 |
| object_name    |   33358 |    18858 | FENGYUN 1C DEB   |   2340 |
| satellite_name |    5505 |     5492 | Starlink-2213    |      2 |
| official_name  |    5505 |     5483 | Jilin-1          |      5 |
| object_type    |   33358 |        4 | PAYLOAD          |  18339 |
| category       |   33358 |        5 | Active Satellite |  13106 |
| ops_status     |   33358 |        7 | UNKNOWN          |  16567 |
| data_status    |     944 |        2 | NEA              |    943 |
| owner_code     |   33358 |      105 | US               |  17083 |
| launch_id      |   33358 |     3939 | 1999-025         |   2346 |

### 📈 Numeric Overview
|          |   count |    mean |     std |   min |     25% |     50% |     75% |   max |
|:---------|--------:|--------:|--------:|------:|--------:|--------:|--------:|------:|
| norad_id |   33358 | 42670.6 | 18778.5 |     5 | 28912.2 | 44884.5 | 59505.8 | 68379 |
| in_orbit |   33358 |     1   |     0   |     1 |     1   |     1   |     1   |     1 |

# Table: orbital_data

**Dimensions:** 33,358 rows × 16 columns

**Memory Footprint:** 8.59 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **orbit_class** | `object` | 0 | 100.0% | ✅ |
| **orbit_type** | `object` | 0 | 100.0% | ✅ |
| **period_minutes** | `float64` | 0 | 100.0% | ✅ |
| **perigee_km** | `float64` | 0 | 100.0% | ✅ |
| **apogee_km** | `float64` | 0 | 100.0% | ✅ |
| **inclination_degrees** | `float64` | 0 | 100.0% | ✅ |
| **eccentricity** | `float64` | 0 | 100.0% | ✅ |
| **semi_major_axis_km** | `float64` | 0 | 100.0% | ✅ |
| **launch_mass_kg** | `float64` | 27,853 | 16.5% | ⚠️ |
| **proxy_mass_kg** | `float64` | 0 | 100.0% | ✅ |
| **dry_mass_kg** | `float64` | 0 | 100.0% | ✅ |
| **power_watts** | `float64` | 27,853 | 16.5% | ⚠️ |
| **proxy_power_watts** | `float64` | 0 | 100.0% | ✅ |
| **rcs** | `float64` | 0 | 100.0% | ✅ |
| **rcs_class** | `object` | 0 | 100.0% | ✅ |

### 📝 Object Overview
|             |   count |   unique | top        |   freq |
|:------------|--------:|---------:|:-----------|-------:|
| orbit_class |   33358 |        5 | LEO        |  27302 |
| orbit_type  |   33358 |        5 | Other/Misc |  28490 |
| rcs_class   |   33358 |        4 | UNKNOWN    |  18639 |

### 📈 Numeric Overview
|                     |   count |          mean |          std |         min |            25% |            50% |            75% |           max |
|:--------------------|--------:|--------------:|-------------:|------------:|---------------:|---------------:|---------------:|--------------:|
| norad_id            |   33358 | 42670.6       | 18778.5      |    5        | 28912.2        | 44884.5        | 59505.8        |  68379        |
| period_minutes      |   33358 |   240.796     |   709.497    |   88.56     |    94.32       |    99.83       |   109.46       |  55699.4      |
| perigee_km          |   33358 |  3160.82      |  9013.94     |  102        |   482          |   626          |   915          | 314973        |
| apogee_km           |   33358 |  5815.7       | 15931.1      |  206        |   487          |   786          |  1302          | 641287        |
| inclination_degrees |   33358 |    66.4617    |    28.2341   |    0        |    50          |    70          |    97.38       |    149.64     |
| eccentricity        |   33358 |     0.0564723 |     0.167341 |   -0.725044 |     0.00014611 |     0.00130404 |     0.00885031 |      0.972663 |
| semi_major_axis_km  |   33358 | 10867.5       | 11481.3      | 6581.4      |  6863.77       |  7128.54       |  7579.91       | 483126        |
| launch_mass_kg      |    5505 |   878.035     |  6251.37     |    1        |   227          |   260          |   290          | 450000        |
| proxy_mass_kg       |   33358 |   444.121     |  2591.27     |    1        |    50          |   290          |   355          | 450000        |
| dry_mass_kg         |   33358 |   382.169     |  2307.07     |    0.9      |    50          |   234          |   319.5        | 405000        |
| power_watts         |    5505 |  1228.67      |  3220.29     |    0        |   120          |   120          |   815          |  84000        |
| proxy_power_watts   |   33358 |   281.143     |  1380.12     |    0        |     0          |   120          |   163.846      |  84000        |
| rcs                 |   33358 |     1.82635   |     9.444    |    0.0001   |     0.0181     |     1          |     1          |    830.035    |

# Table: ucs_details

**Dimensions:** 33,358 rows × 7 columns

**Memory Footprint:** 4.99 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **lifetime_years** | `float64` | 27,853 | 16.5% | ⚠️ |
| **sat_age_years** | `int64` | 0 | 100.0% | ✅ |
| **primary_purpose** | `object` | 12,851 | 61.5% | ⚠️ |
| **detailed_purpose** | `object` | 27,853 | 16.5% | ⚠️ |
| **geo_longitude** | `float64` | 0 | 100.0% | ✅ |
| **un_registry** | `object` | 12,851 | 61.5% | ⚠️ |

### 📝 Object Overview
|                  |   count |   unique | top            |   freq |
|:-----------------|--------:|---------:|:---------------|-------:|
| primary_purpose  |   20507 |        7 | Not Applicable |  15002 |
| detailed_purpose |    5505 |       43 | Not Specified  |   4799 |
| un_registry      |   20507 |       62 | Not Applicable |  15002 |

### 📈 Numeric Overview
|                |   count |         mean |         std |     min |     25% |     50% |     75% |   max |
|:---------------|--------:|-------------:|------------:|--------:|--------:|--------:|--------:|------:|
| norad_id       |   33358 | 42670.6      | 18778.5     |    5    | 28912.2 | 44884.5 | 59505.8 | 68379 |
| lifetime_years |    5505 |     5.63465  |     3.58569 |    0.25 |     4   |     4   |     5   |    30 |
| sat_age_years  |   33358 |    19.283    |    19.297   |    0    |     2   |    10.5 |    34   |    68 |
| geo_longitude  |   33358 |     0.388015 |    12.536   | -179.8  |     0   |     0   |     0   |   359 |

# Table: risk_assessment

**Dimensions:** 33,358 rows × 4 columns

**Memory Footprint:** 1.02 MB

**Primary Key Check**: ✅ No duplicate norad_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **norad_id** | `int64` | 0 | 100.0% | ✅ |
| **velocity_kms** | `float64` | 0 | 100.0% | ✅ |
| **kinetic_joules** | `float64` | 0 | 100.0% | ✅ |
| **is_zombie** | `int64` | 0 | 100.0% | ✅ |

### 📈 Numeric Overview
|                |   count |            mean |             std |         min |             25% |             50% |             75% |             max |
|:---------------|--------:|----------------:|----------------:|------------:|----------------:|----------------:|----------------:|----------------:|
| norad_id       |   33358 | 42670.6         | 18778.5         | 5           | 28912.2         | 44884.5         | 59505.8         | 68379           |
| velocity_kms   |   33358 |     6.91903     |     1.36957     | 0.908319    |     7.25165     |     7.47771     |     7.62057     |     7.78233     |
| kinetic_joules |   33358 |     8.87036e+09 |     7.36245e+10 | 2.81882e+07 |     1.38271e+09 |     7.49032e+09 |     1.03116e+10 |     1.31907e+13 |
| is_zombie      |   33358 |     0.156874    |     0.363687    | 0           |     0           |     0           |     0           |     1           |

# Table: ownership_operators

**Dimensions:** 105 rows × 10 columns

**Memory Footprint:** 0.04 MB

**Primary Key Check**: ✅ No duplicate owner_code values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **owner_code** | `object` | 0 | 100.0% | ✅ |
| **owner** | `object` | 0 | 100.0% | ✅ |
| **country_operator** | `object` | 24 | 77.1% | ⚠️ |
| **users** | `object` | 24 | 77.1% | ⚠️ |
| **is_commercial** | `int64` | 0 | 100.0% | ✅ |
| **is_government** | `int64` | 0 | 100.0% | ✅ |
| **is_military** | `int64` | 0 | 100.0% | ✅ |
| **is_civil** | `int64` | 0 | 100.0% | ✅ |
| **contractor** | `object` | 24 | 77.1% | ⚠️ |
| **contractor_country** | `object` | 24 | 77.1% | ⚠️ |

### 📝 Object Overview
|                    |   count |   unique | top                 |   freq |
|:-------------------|--------:|---------:|:--------------------|-------:|
| owner_code         |     105 |      105 | AB                  |      1 |
| owner              |     105 |      105 | AB                  |      1 |
| country_operator   |      81 |       64 | MULTINATIONAL       |      8 |
| users              |      81 |       10 | Government          |     28 |
| contractor         |      81 |       54 | Thales Alenia Space |      7 |
| contractor_country |      81 |       33 | USA                 |     23 |

### 📈 Numeric Overview
|               |   count |     mean |      std |   min |   25% |   50% |   75% |   max |
|:--------------|--------:|---------:|---------:|------:|------:|------:|------:|------:|
| is_commercial |     105 | 0.52381  | 0.501828 |     0 |     0 |     1 |     1 |     1 |
| is_government |     105 | 0.514286 | 0.502193 |     0 |     0 |     1 |     1 |     1 |
| is_military   |     105 | 0.27619  | 0.449257 |     0 |     0 |     0 |     1 |     1 |
| is_civil      |     105 | 0.27619  | 0.449257 |     0 |     0 |     0 |     1 |     1 |

# Table: launch_events

**Dimensions:** 3,939 rows × 4 columns

**Memory Footprint:** 0.69 MB

**Primary Key Check**: ✅ No duplicate launch_id values detected


---
### 📊 Data Quality Audit
| Column | Type | Nulls | Fill % | Status |
| :--- | :--- | :--- | :--- | :--- |
| **launch_id** | `object` | 0 | 100.0% | ✅ |
| **launch_date** | `object` | 0 | 100.0% | ✅ |
| **launch_year** | `int64` | 0 | 100.0% | ✅ |
| **launch_site** | `object` | 0 | 100.0% | ✅ |

### 📝 Object Overview
|             |   count |   unique | top        |   freq |
|:------------|--------:|---------:|:-----------|-------:|
| launch_id   |    3939 |     3939 | 1958-002   |      1 |
| launch_date |    3939 |     3537 | 2022-08-04 |      5 |
| launch_site |    3939 |       59 | AFETR      |    737 |

### 📈 Numeric Overview
|             |   count |    mean |     std |   min |   25% |   50% |   75% |   max |
|:------------|--------:|--------:|--------:|------:|------:|------:|------:|------:|
| launch_year |    3939 | 2002.25 | 19.0305 |  1958 |  1986 |  2005 |  2021 |  2026 |


SQLite build complete: ../data/clean/orbital_debris.db
